# ETP Primitives Deep Dive

## Introduction

ETP (Eligibility Trace Propagation) primitives are JAX custom primitives that **mark weight operations** in the computational graph. They replace the old `ETraceOp` / JIT-name-matching system with a cleaner, more robust approach.

Key design principles:

- **Type identity, not string matching.** The compiler identifies ETP primitives by checking `eqn.primitive in ETP_PRIMITIVES` -- a set membership test on the primitive object itself. This is more reliable than the old approach of matching JIT function names.

- **All JAX rules are auto-derived.** Each primitive's `impl` delegates to standard JAX ops (e.g., `x @ w`, `jax.lax.conv_general_dilated`). The `register_primitive()` helper automatically derives JIT, grad, vmap, JVP, transpose, abstract_eval, and lowering rules from the implementation. No hand-written derivative formulas are needed.

- **Only ETP-specific rules need hand-writing.** Two (optionally three) rule registries capture the online-learning-specific logic:
  - `etp_rules_yw_to_w` -- D-RTRL trace propagation
  - `etp_rules_xy_to_dw` -- weight gradient computation
  - `etp_rules_init_state` -- trace state initialization

- **Primitive-based parameter selection.** A parameter participates in ETP if and only if it flows through an ETP primitive (`braintrace.matmul`, `braintrace.element_wise`, etc.). Parameters used with regular JAX ops are automatically excluded -- no special parameter class is needed.

In [ ]:
import jax
import jax.numpy as jnp

import braintrace

## The Five Primitive Functions

`braintrace` provides five user-facing ETP primitive functions:

| Function | Underlying primitives | Purpose |
|---|---|---|
| `braintrace.matmul` | `etp_mm_p` (batched) / `etp_mv_p` (unbatched) | Dense matrix multiplication |
| `braintrace.element_wise` | `etp_elemwise_p` | Element-wise (diagonal) weight ops |
| `braintrace.conv` | `etp_conv_p` | Convolution |
| `braintrace.sparse_matmul` | `etp_sp_mm_p` / `etp_sp_mv_p` | Sparse matrix multiplication |
| `braintrace.lora_matmul` | `etp_lora_mm_p` / `etp_lora_mv_p` | LoRA (Low-Rank Adaptation) matmul |

Each function auto-dispatches between batched and unbatched variants based on input dimensionality.

### 1. `braintrace.matmul(x, weight, bias=None)` -- Dense Matrix Multiplication

Computes $y = x \, @ \, w \; (+ b)$.

Auto-dispatches based on `x.ndim`:
- `x.ndim >= 2` --> `etp_mm_p` (batched): expects `x` of shape `(batch, in_features)`
- `x.ndim == 1` --> `etp_mv_p` (unbatched): expects `x` of shape `(in_features,)`

In [ ]:
# Batched matmul: x has shape (batch, in_features)
x_batched = jnp.ones((4, 3))    # batch=4, in_features=3
w = jnp.ones((3, 5))            # in_features=3, out_features=5

y_batched = braintrace.matmul(x_batched, w)
print("Batched output shape:", y_batched.shape)   # (4, 5)

# Unbatched matmul: x has shape (in_features,)
x_single = jnp.ones((3,))       # in_features=3

y_single = braintrace.matmul(x_single, w)
print("Unbatched output shape:", y_single.shape)   # (5,)

In [ ]:
# With bias
b = jnp.zeros((5,))

y_with_bias = braintrace.matmul(x_batched, w, bias=b)
print("With bias:", y_with_bias.shape)              # (4, 5)

### 2. `braintrace.element_wise(weight, fn=None)` -- Element-wise Operation

Applies an optional function `fn` to the weight and passes the result through a marker primitive. The operation is treated as *diagonal* in the hidden-state space.

$$y = \texttt{fn}(w)$$

This is commonly used for:
- Gating mechanisms in RNNs (learnable gate biases)
- Learnable time constants or thresholds in spiking neural networks
- Any parameter that enters the computation element-wise

In [ ]:
# Identity (default fn): just marks the weight for ETP
w_gate = jnp.array([0.5, -0.3, 0.8, 0.1])

y_identity = braintrace.element_wise(w_gate)
print("Identity:", y_identity)

# With a transformation function
y_sigmoid = braintrace.element_wise(w_gate, fn=jax.nn.sigmoid)
print("Sigmoid:", y_sigmoid)

# With absolute value (e.g., enforcing positive time constants)
y_abs = braintrace.element_wise(w_gate, fn=jnp.abs)
print("Abs:", y_abs)

### 3. `braintrace.conv(x, kernel, bias=None, *, strides, padding, ...)` -- Convolution

ETP-aware convolution that wraps `jax.lax.conv_general_dilated`. Computes:

$$y = \text{conv}(x, \text{kernel}) \; (+ b)$$

**Important**: Always expects a batch dimension on `x`.

Supports all parameters of `jax.lax.conv_general_dilated`: `strides`, `padding`, `lhs_dilation`, `rhs_dilation`, `feature_group_count`, `batch_group_count`, and `dimension_numbers`.

In [ ]:
# 1D convolution example
# x: (batch, spatial, channels) with dimension_numbers
x_1d = jnp.ones((2, 16, 3))         # batch=2, length=16, in_channels=3
kernel_1d = jnp.ones((4, 3, 8))     # kernel_size=4, in_channels=3, out_channels=8

y_conv = braintrace.conv(
    x_1d, kernel_1d,
    strides=(1,),
    padding='SAME',
    dimension_numbers=('NWC', 'WIO', 'NWC'),
)
print("Conv1D output shape:", y_conv.shape)  # (2, 16, 8)

In [ ]:
# 2D convolution example
x_2d = jnp.ones((2, 32, 32, 3))          # batch=2, H=32, W=32, in_channels=3
kernel_2d = jnp.ones((3, 3, 3, 16))      # kH=3, kW=3, in_channels=3, out_channels=16

y_conv2d = braintrace.conv(
    x_2d, kernel_2d,
    strides=(1, 1),
    padding='SAME',
    dimension_numbers=('NHWC', 'HWIO', 'NHWC'),
)
print("Conv2D output shape:", y_conv2d.shape)  # (2, 32, 32, 16)

### 4. `braintrace.sparse_matmul(x, weight_data, *, sparse_mat, bias=None)` -- Sparse Matmul

ETP-aware sparse matrix multiplication. Computes:

$$y = x \, @ \, \text{sparse}(w) \; (+ b)$$

The `sparse_mat` argument provides the sparse structure (indices), while `weight_data` contains only the non-zero values. This is useful for models with sparse connectivity patterns, such as biologically plausible neural networks or graph neural networks.

In [ ]:
import brainevent

# Create a sparse connectivity matrix
dense_w = jnp.where(
    jax.random.uniform(jax.random.PRNGKey(0), (50, 50)) < 0.1,
    jax.random.normal(jax.random.PRNGKey(1), (50, 50)),
    0.0
)
sparse_mat = brainevent.CSR.fromdense(dense_w)

# The learnable parameter is just the non-zero data
weight_data = sparse_mat.data

x_sp = jnp.ones((4, 50))  # batch=4, features=50
y_sp = braintrace.sparse_matmul(x_sp, weight_data, sparse_mat=sparse_mat)
print("Sparse matmul output shape:", y_sp.shape)  # (4, 50)

### 5. `braintrace.lora_matmul(x, B, A, *, alpha=1.0, bias=None)` -- LoRA Matmul

Low-Rank Adaptation matmul. Computes:

$$y = \alpha \cdot x \, @ \, B \, @ \, A \; (+ b)$$

where $B \in \mathbb{R}^{\text{in} \times \text{rank}}$ and $A \in \mathbb{R}^{\text{rank} \times \text{out}}$ are low-rank factors. This is useful for parameter-efficient fine-tuning of large models, where only the low-rank factors are trained.

In [ ]:
in_features, out_features, rank = 64, 32, 4

B = jax.random.normal(jax.random.PRNGKey(0), (in_features, rank)) * 0.01
A = jax.random.normal(jax.random.PRNGKey(1), (rank, out_features)) * 0.01

x_lora = jnp.ones((8, in_features))  # batch=8

y_lora = braintrace.lora_matmul(x_lora, B, A, alpha=2.0)
print("LoRA output shape:", y_lora.shape)  # (8, 32)
print("LoRA output (first sample):", y_lora[0])

## JAX Compatibility

All ETP primitives are fully compatible with JAX transformations. Since `register_primitive()` auto-derives JIT, grad, vmap, and JVP rules from the implementation, they work seamlessly with the standard JAX API.

In [ ]:
x = jnp.ones((4, 3))
w = jnp.ones((3, 5))

# ---- JIT compilation ----
y_jit = jax.jit(braintrace.matmul)(x, w)
print("JIT output shape:", y_jit.shape)

In [ ]:
# ---- Gradient computation ----
grad_fn = jax.grad(lambda w: jnp.sum(braintrace.matmul(x, w)))
dw = grad_fn(w)
print("Gradient shape:", dw.shape)
print("Gradient values:\n", dw)

In [ ]:
# ---- Vectorized mapping (vmap) ----
# vmap over a batch of inputs, each of shape (4, 3)
xs = jnp.ones((8, 4, 3))  # 8 different batches
vmap_fn = jax.vmap(lambda x_i: braintrace.matmul(x_i, w))
ys = vmap_fn(xs)
print("vmap output shape:", ys.shape)  # (8, 4, 5)

In [ ]:
# ---- JVP (forward-mode differentiation) ----
primals = (x, w)
tangents = (jnp.ones_like(x), jnp.ones_like(w))

y_primal, y_tangent = jax.jvp(braintrace.matmul, primals, tangents)
print("JVP primal shape:", y_primal.shape)
print("JVP tangent shape:", y_tangent.shape)

In [ ]:
# ---- Composability: JIT + grad + vmap ----
@jax.jit
def batched_grad(xs, w):
    """Compute per-sample gradients w.r.t. the weight."""
    def single_grad(x_i):
        return jax.grad(lambda w_: jnp.sum(braintrace.matmul(x_i, w_)))(w)
    return jax.vmap(single_grad)(xs)

xs = jnp.ones((8, 4, 3))
per_sample_grads = batched_grad(xs, w)
print("Per-sample gradients shape:", per_sample_grads.shape)  # (8, 3, 5)

## Argument Conventions

Each ETP primitive follows specific conventions for its input variables (`invars`) and static parameters. Understanding these conventions is essential when working with the compiler or adding custom primitives.

| Primitive | `invars[0]` | `invars[1]` | `invars[2]` | `invars[3]` | Static params |
|---|---|---|---|---|---|
| `etp_mm_p` / `etp_mv_p` | input `x` | weight `w` | bias `b` (opt) | -- | `has_bias` |
| `etp_elemwise_p` | processed `y` | -- | -- | -- | (none) |
| `etp_conv_p` | input `x` | kernel `w` | bias `b` (opt) | -- | `has_bias`, `strides`, `padding`, ... |
| `etp_sp_mm_p` / `etp_sp_mv_p` | input `x` | weight data | bias `b` (opt) | -- | `sparse_mat`, `has_bias` |
| `etp_lora_mm_p` / `etp_lora_mv_p` | input `x` | matrix `B` | matrix `A` | bias `b` (opt) | `alpha`, `has_bias` |

Notes:
- The weight variable is always at index 1 for matmul/conv/sparse/lora, and index 0 for elemwise.
- The `has_bias` flag is a static parameter (not a traced value) that controls whether the optional bias argument is present.
- For convolution, all `jax.lax.conv_general_dilated` parameters are passed as static params.

## Rule Registries

ETP uses three global dictionaries to store operation-specific rules. These are the *only* things that need hand-writing -- all standard JAX rules are auto-derived.

### `etp_rules_yw_to_w` -- D-RTRL Trace Propagation

Signature: `(hidden_dim, trace, **params) -> updated_trace`

Defines how the eligibility trace is propagated through the operation. Called during the D-RTRL trace update step.

### `etp_rules_xy_to_dw` -- Weight Gradient Computation

Signature: `(x, hidden_dim, w, **params) -> dw`

Computes the weight gradient (or immediate influence term) for both D-RTRL and ES-D-RTRL algorithms.

### `etp_rules_init_state` -- Trace State Initialization

Signature: `(x_var, y_var, weight, num_hidden_state) -> zeros`

Creates the initial (zero) trace state. The shape depends on whether the primitive is batched or unbatched.

In [ ]:
from braintrace._etrace_operators import (
    etp_rules_yw_to_w,
    etp_rules_xy_to_dw,
    etp_rules_init_state,
    ETP_PRIMITIVES,
    BATCHED_PRIMITIVES,
)

print("All ETP primitives:")
for p in sorted(ETP_PRIMITIVES, key=lambda p: p.name):
    batched_tag = " [batched]" if p in BATCHED_PRIMITIVES else ""
    print(f"  {p.name}{batched_tag}")

print("\nTrace propagation rules (yw_to_w):")
for p in sorted(etp_rules_yw_to_w.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

print("\nWeight gradient rules (xy_to_dw):")
for p in sorted(etp_rules_xy_to_dw.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

print("\nInit state rules:")
for p in sorted(etp_rules_init_state.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

## Adding a Custom Primitive

Adding a new ETP primitive takes only a few steps. Here we create a scaled matrix multiplication primitive as an example:

$$y = \text{scale} \cdot (x \, @ \, w)$$

In [ ]:
from braintrace._etrace_operators import (
    register_primitive,
    etp_rules_yw_to_w,
    etp_rules_xy_to_dw,
    etp_rules_init_state,
)


# Step 1: Define the implementation
# This is a plain JAX function -- no special annotations needed.
def _scaled_matmul_impl(x, w, *, scale=1.0, has_bias=False):
    return scale * (x @ w)


# Step 2: Register as an ETP primitive
# register_primitive() auto-derives all JAX rules (jit, grad, vmap, jvp)
# from the implementation function.
scaled_mm_p = register_primitive('etp_scaled_mm', _scaled_matmul_impl, batched=True)

print("Primitive registered:", scaled_mm_p)
print("Is ETP primitive:", scaled_mm_p in ETP_PRIMITIVES)
print("Is batched:", scaled_mm_p in BATCHED_PRIMITIVES)

In [ ]:
# Step 3: Register ETP-specific rules

# Trace propagation: how the eligibility trace updates through this op
etp_rules_yw_to_w[scaled_mm_p] = lambda hidden_dim, trace, **p: (
    trace * jnp.expand_dims(hidden_dim, axis=1)
)

# Weight gradient: immediate influence of weight on output
etp_rules_xy_to_dw[scaled_mm_p] = lambda x, hidden_dim, w, **p: (
    jax.vjp(lambda w_: p.get('scale', 1.0) * (x @ w_), w)[1](hidden_dim)[0]
)

# Trace state initialization: create zero-filled trace of the right shape
etp_rules_init_state[scaled_mm_p] = lambda x_var, y_var, weight, n: (
    jnp.zeros((x_var.aval.shape[0], *jnp.shape(weight.value), n))
)

print("All three rules registered.")

In [ ]:
# Step 4: Use the custom primitive

x = jnp.ones((4, 3))
w = jnp.ones((3, 5))

# Direct usage via primitive.bind()
y = scaled_mm_p.bind(x, w, scale=2.0, has_bias=False)
print("Output shape:", y.shape)
print("Output (scale=2.0):\n", y)

# Verify: should equal 2.0 * (x @ w)
y_expected = 2.0 * (x @ w)
print("\nMatches expected:", jnp.allclose(y, y_expected))

In [ ]:
# The custom primitive works with all JAX transformations automatically

# JIT
y_jit = jax.jit(lambda x, w: scaled_mm_p.bind(x, w, scale=2.0, has_bias=False))(x, w)
print("JIT works:", jnp.allclose(y_jit, y_expected))

# Grad
dw = jax.grad(lambda w: jnp.sum(scaled_mm_p.bind(x, w, scale=2.0, has_bias=False)))(w)
print("Grad shape:", dw.shape)

# Vmap
xs = jnp.ones((8, 4, 3))
ys = jax.vmap(lambda xi: scaled_mm_p.bind(xi, w, scale=2.0, has_bias=False))(xs)
print("Vmap output shape:", ys.shape)

## Summary

ETP primitives provide a clean, extensible foundation for online learning in recurrent networks:

- **5 built-in primitives** cover the most common use cases: dense matmul, element-wise ops, convolution, sparse matmul, and LoRA.

- **Custom primitives can be added in approximately 10 lines of code**: define an implementation function, call `register_primitive()`, and register the three ETP-specific rules.

- **All JAX transformations (JIT, grad, vmap, JVP) work automatically** -- only the online-learning-specific rules need hand-writing.

- **Parameter selection is primitive-based**: use `braintrace.matmul` to include a weight in ETP, or plain `x @ w` to exclude it. No special parameter class required.

For more details on how the compiler uses these primitives to build the computational graph, see the [ETP Graph Compilation](../advanced/IR_analysis-en.ipynb) tutorial.